# 01 — Data Cleaning

Loads the Health & Lifestyle survey, renames columns, computes BMI, derives composite variables (sleep quality, stress score, recovery), and saves `data/processed/combined_data.csv`.

In [91]:
import pandas as pd
import numpy as np

RAW_PATH    = '../data/raw/Health & Lifestyle Questionnaire (Responses) - Form responses 1.csv'
OUTPUT_PATH = '../data/processed/combined_data.csv'

raw = pd.read_csv(RAW_PATH)
print('Raw shape:', raw.shape)
raw.head(2)

Raw shape: (25, 36)


,Timestamp,"Consent to Participate \nBy selecting “I agree,” you confirm that:\n - You are 18 years or older\n- You voluntarily agree to participate and for your responses to be used for analysis",Enter your full name,Age,Gender,Please indicate your height (cm):,Please indicate your weight (kg):,Indicate your experience level for each activity (choose ‘I do not practice this’ if it doesn’t apply) [Weight lifting],Indicate your experience level for each activity (choose ‘I do not practice this’ if it doesn’t apply) [Yoga/Pilates],Indicate your experience level for each activity (choose ‘I do not practice this’ if it doesn’t apply) [Barre],...,Average nightly sleep duration:,"If you use a wearable device, please provide your average REM and deep sleep duration:","How would you rate the overall quality of your sleep. Think about how quickly you fall asleep, how often you wake up, and how rested you feel in the morning.",How often do you wake up feeling physically and mentally refreshed?,"Perceived stress level over the past 3 months: (Think about how often you feel stressed, how intense it feels, and whether it affects sleep, mood, or your body.)",How often do you feel mentally overwhelmed by daily responsibilities?,"How often does stress affect your physical state (Physical effects can include tight shoulders/neck/jaw, stiffness, headaches, fatigue, shallow breathing, or stomach discomfort.)?","How often do you intentionally use physical recovery practices? (e.g., mobility work, stretching, foam rolling, massage, light movement, cold/heat therapy)","How often do you intentionally use mental recovery practices? (e.g., breathwork, meditation, mindfulness, relaxation, journaling, time outdoors, screen-free downtime, social activities)",Have you experienced any improvement in your mental health or overall well-being due to the physical activities you practice?
0,02/03/2026 16:56:21,I agree,P16,30-39,Male,187,90.0,> 5 years,I do not practice this,I do not practice this,...,6–8 hours,yes,Fair,Often,Moderate (Regular stress; sometimes affects fo...,I feel overwhelmed a few times per month and i...,Sometimes (a few times per month),About every 2–3 weeks,About every 2–3 weeks,Slightly
1,04/03/2026 18:35:27,I agree,P03,18-29,Female,167,62.0,< 1 year,1-3 years,< 1 year,...,6–8 hours,3 hours,Good,Sometimes,High (Stressed many days; frequent tension or ...,I feel overwhelmed a few times per month and i...,Sometimes (a few times per month),2–3 times per week,About every 2–3 weeks,Very significantly


## Step 1 — Rename columns

In [92]:
# Map Google Forms headers to short names
SHORT_NAMES = [
    'timestamp',
    'consent',
    'name',
    'age_raw',
    'gender_raw',
    'height_cm',
    'weight_kg',
    'exp_weightlifting',
    'exp_yoga',
    'exp_barre',
    'exp_running',
    'exp_crossfit',
    'exp_cycling',
    'exp_other',
    'exp_other_specify',
    'freq_flexibility',
    'freq_strength',
    'hours_per_week_raw',
    'end_range_control_raw',
    'dominant_training_raw',
    'injury_raw',
    'injury_types',
    'chronic_pain_raw',
    'daily_sitting_raw',
    'sitting_breaks_raw',
    'daily_activity_raw',
    'sleep_duration_raw',
    'sleep_wearable',
    'sleep_quality_raw',
    'wake_refreshed_raw',
    'stress_level_raw',
    'overwhelm_raw',
    'stress_physical_raw',
    'freq_physical_recovery_raw',
    'freq_mental_recovery_raw',
    'wellbeing_improvement',
    '_unused'
]

df = raw.copy()
df.columns = SHORT_NAMES[:len(df.columns)]

# Keep only consented rows with a valid timestamp
df = df[
    df['consent'].str.strip() == 'I agree'
].copy()

# Drop artifact row (last row is a stray note)
df = df[df['timestamp'].str.match(r'\d{2}/\d{2}/\d{4}', na=False)].copy()
df = df.reset_index(drop=True)

print(f'Valid participants: {len(df)}')
df[['name', 'age_raw', 'gender_raw']].head()

Valid participants: 25


,name,age_raw,gender_raw
0,P16,30-39,Male
1,P03,18-29,Female
2,P24,50-59,Male
3,P22,50-59,Female
4,P06,18-29,Female


## Step 2 — Numeric base fields

In [93]:
df['height_cm'] = pd.to_numeric(df['height_cm'], errors='coerce')
df['weight_kg'] = pd.to_numeric(df['weight_kg'], errors='coerce')

## Variable 3 — BMI
**Continuous** — computed from self-reported height (cm) and weight (kg).
Height and weight are entered as direct values in this survey, so BMI = weight / (height/100)².

In [94]:
df['bmi'] = (df['weight_kg'] / (df['height_cm'] / 100) ** 2).round(1)
df[['name', 'height_cm', 'weight_kg', 'bmi']]

,name,height_cm,weight_kg,bmi
0,P16,187,90.0,25.7
1,P03,167,62.0,22.2
2,P24,180,77.0,23.8
3,P22,165,62.0,22.8
4,P06,172,60.0,20.3
5,P12,179,55.0,17.2
6,P04,164,50.0,18.6
7,P19,179,80.0,25.0
8,P21,167,57.0,20.4
9,P05,166,56.0,20.3


## Training category
**Nominal** — derived from three questions: flexibility frequency, strength frequency, and weekly volume.

Classification applies in order:

| Priority | Category | Criteria |
|---|---|---|
| 1 | Low Active | Volume < 2 hrs (regardless of frequency) |
| 2 | Hybrid | Volume ≥ 2 hrs AND Strength ≥2×/week AND Flexibility ≥2×/week |
| 3 | Strength-dominant | Volume ≥ 2 hrs AND Strength ≥2×/week AND Flexibility ≤1×/week |
| 4 | Flexibility-dominant | Volume ≥ 2 hrs AND Flexibility ≥2×/week AND Strength ≤1×/week |
| 5 | Unclassified | Does not meet any of the above criteria |

In [95]:
LOW_FREQ = {'Never', 'Rarely (≤1×/week)'}
HIGH_FREQ = {'Sometimes (2–3×/week)', 'Often (4–5×/week)', 'Very often (≥6×/week)'}
LOW_VOL   = {'Less than 2 hours'}

def classify_training(row):
    str_freq  = row['freq_strength']
    flex_freq = row['freq_flexibility']
    volume    = row['hours_per_week_raw']

    is_str_low   = str_freq  in LOW_FREQ
    is_str_high  = str_freq  in HIGH_FREQ
    is_flex_low  = flex_freq in LOW_FREQ
    is_flex_high = flex_freq in HIGH_FREQ

    # 1. Volume is the primary filter for Low Active
    if volume in LOW_VOL:
        return 'Low Active'
    # 2. Hybrid
    if is_str_high and is_flex_high:
        return 'Hybrid'
    # 3. Strength-dominant
    if is_str_high and is_flex_low:
        return 'Strength-dominant'
    # 4. Flexibility-dominant
    if is_flex_high and is_str_low:
        return 'Flexibility-dominant'

    return 'Unclassified'

df['training_category'] = df.apply(classify_training, axis=1)
print(df['training_category'].value_counts())
print()
print(df[['name', 'freq_strength', 'freq_flexibility', 'hours_per_week_raw', 'training_category']])

training_category
Hybrid                  9
Flexibility-dominant    7
Strength-dominant       5
Low Active              2
Unclassified            2
Name: count, dtype: int64

    name          freq_strength       freq_flexibility  hours_per_week_raw  \
0    P16  Sometimes (2–3×/week)      Rarely (≤1×/week)           2–5 hours   
1    P03  Sometimes (2–3×/week)  Sometimes (2–3×/week)           2–5 hours   
2    P24      Rarely (≤1×/week)                  Never   Less than 2 hours   
3   P22       Rarely (≤1×/week)  Sometimes (2–3×/week)           2–5 hours   
4    P06  Sometimes (2–3×/week)      Often (4–5×/week)          5–10 hours   
5    P12  Sometimes (2–3×/week)      Rarely (≤1×/week)          5–10 hours   
6   P04       Rarely (≤1×/week)  Very often (≥6×/week)  More than 10 hours   
7    P19      Rarely (≤1×/week)                  Never           2–5 hours   
8   P21   Sometimes (2–3×/week)  Sometimes (2–3×/week)          5–10 hours   
9   P05       Rarely (≤1×/week)  Sometimes (2

## Manual reclassification overrides


In [96]:
OVERRIDES = {
    'P25': 'Flexibility-dominant',
    'P19': 'Strength-dominant',
    'P18': 'Low Active',
    'P22 ': 'Hybrid',
}


VALID_CATEGORIES = {'Hybrid', 'Flexibility-dominant', 'Strength-dominant', 'Low Active', 'Unclassified'}

for name, cat in OVERRIDES.items():
    assert cat in VALID_CATEGORIES, f"Invalid category '{cat}' for '{name}'"
    mask = df['name'] == name
    assert mask.any(), f"Name not found: '{name}'"
    old = df.loc[mask, 'training_category'].values[0]
    df.loc[mask, 'training_category'] = cat
    print(f"  {name}: {old} → {cat}")

if not OVERRIDES:
    print("No overrides applied.")

print()
print(df['training_category'].value_counts())

  P25: Unclassified → Flexibility-dominant
  P19: Unclassified → Strength-dominant
  P18: Strength-dominant → Low Active
  P22 : Flexibility-dominant → Hybrid

training_category
Hybrid                  10
Flexibility-dominant     7
Strength-dominant        5
Low Active               3
Name: count, dtype: int64


## Sleep quality (0–10)
Composite of two items, each coded 0–4, sum scaled ×1.25:
1. Overall sleep quality rating
2. Frequency of waking refreshed

In [97]:
SLEEP_Q_MAP = {
    'Very poor (hard to fall asleep and/or wake up often; wake up tired most days)': 0,
    'Poor': 1, 'Fair': 2, 'Good': 3,
    'Very good(fall asleep easily, sleep through most nights; wake up rested most days)': 4
}
REFRESHED_MAP = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Always': 4}

df['sleep_quality'] = (
    (df['sleep_quality_raw'].map(SLEEP_Q_MAP) + df['wake_refreshed_raw'].map(REFRESHED_MAP)) * 1.25
).round(1)

print(df['sleep_quality'].describe().round(2))

count    25.00
mean      6.20
std       2.35
min       0.00
25%       5.00
50%       6.20
75%       7.50
max      10.00
Name: sleep_quality, dtype: float64


## Recovery practices (1–6 each)

Physical and mental recovery frequency:
Each coded 1–6: Never=1, Once a month or less=2, Every 2–3 weeks=3, Once a week=4, 2–3×/week=5, 4+×/week=6

In [98]:
RECOVERY_MAP = {
    'Never': 1,
    'About once a month or less': 2,
    'About every 2\u20133 weeks': 3,
    'About once a week': 4,
    '2\u20133 times per week': 5,
    '4+ times per week': 6
}

df['physical_recovery'] = df['freq_physical_recovery_raw'].map(RECOVERY_MAP)
df['mental_recovery']   = df['freq_mental_recovery_raw'].map(RECOVERY_MAP)

print('Physical recovery:')
print(df['physical_recovery'].describe().round(2))
print()
print('Mental recovery:')
print(df['mental_recovery'].describe().round(2))

Physical recovery:
count    25.00
mean      3.44
std       1.50
min       1.00
25%       2.00
50%       4.00
75%       5.00
max       6.00
Name: physical_recovery, dtype: float64

Mental recovery:
count    25.00
mean      4.40
std       1.53
min       1.00
25%       3.00
50%       5.00
75%       6.00
max       6.00
Name: mental_recovery, dtype: float64


## Stress Score (0–4)
Composite mean of three items (each 0–4, higher = more stressed):
1. Perceived stress level over past 3 months
2. Frequency of mental overwhelm
3. Frequency of physical stress effects

In [99]:
STRESS_LEVEL_MAP = {
    'Low (Occasional stress on busy days; recovers quickly)': 0,
    'Moderate (Regular stress; sometimes affects focus or sleep)': 1,
    'High (Stressed many days; frequent tension or poor sleep)': 2,
    'Very high (Stressed most days; hard to cope; clear physical/mood effects)': 3
}
OVERWHELM_MAP = {
    'I generally manage responsibilities without feeling overwhelmed.': 0,
    'I feel overwhelmed mainly during peak stress periods (deadlines, exams, family issues).': 1,
    'I feel overwhelmed a few times per month and it affects my focus or productivity.': 2,
    'I feel overwhelmed weekly and often struggle to keep up with daily tasks.': 3,
    'I feel overwhelmed most days and it frequently affects sleep, mood, or functioning.': 4
}
STRESS_PHYS_MAP = {
    'Rarely (only during very stressful situations)': 0,
    'Sometimes (a few times per month)': 1,
    'Often (about 1\u20133 times per week)': 2,
    'Always (most days or almost every day)': 3
}

df['stress_score'] = pd.DataFrame({
    's1': df['stress_level_raw'].map(STRESS_LEVEL_MAP),
    's2': df['overwhelm_raw'].map(OVERWHELM_MAP),
    's3': df['stress_physical_raw'].map(STRESS_PHYS_MAP),
}).mean(axis=1).round(2)

print(df['stress_score'].describe().round(2))

count    25.00
mean      1.19
std       0.76
min       0.00
25%       0.67
50%       1.33
75%       1.67
max       3.00
Name: stress_score, dtype: float64


## Sedentary Score (−5 to 11)
`Sedentary Score = Sitting score + Break score + Behavior risk score`

| Component | Encoding |
|---|---|
| Sitting score | <4h=1, 4–6h=2, 6–8h=3, >8h=4 |
| Break score | ≥every 30min=1, 30–60min=2, 1–2h=3, 2–4h=4, <every 4h=5 |
| Behavior risk | +1 per sedentary indicator, −1 per movement indicator |

In [100]:
SITTING_MAP = {
    'Less than 4 hours': 1,
    '4\u20136 hours': 2,
    '6\u20138 hours': 3,
    'More than 8 hours': 4
}
BREAK_MAP = {
    'About once every 30 minutes or more often': 1,
    'About once every 30\u201360 minutes': 2,
    'About once every 1\u20132 hours': 3,
    'About once every 2\u20134 hours': 4,
    'Less than once every 4 hours': 5
}
SEDENTARY_IND = ['Travel mainly by car/motorbike', 'Work/study mainly from home']
MOVEMENT_IND = [
    'Walk or cycle most days',
    'Use public transport most days',
    'Take stairs regularly instead of elevator',
    'Do short \u201cmicro-workouts\u201d',
    'Take short walking breaks',
    'Do household/errand activity most days',
    'Do regular lifting/carrying'
]

def behavior_risk(s):
    if pd.isna(s):
        return 0
    sl = s.lower()
    return (
        sum(1 for i in SEDENTARY_IND if i.lower() in sl)
        - sum(1 for i in MOVEMENT_IND if i.lower() in sl)
    )

df['sedentary_score'] = (
    df['daily_sitting_raw'].map(SITTING_MAP)
    + df['sitting_breaks_raw'].map(BREAK_MAP)
    + df['daily_activity_raw'].apply(behavior_risk)
)

print('Range:', df['sedentary_score'].min(), 'to', df['sedentary_score'].max())
print(df['sedentary_score'].describe().round(2))

Range: 0 to 8
count    25.00
mean      3.12
std       2.15
min       0.00
25%       1.00
50%       3.00
75%       5.00
max       8.00
Name: sedentary_score, dtype: float64


## Sitting Time (hours/day)
**Continuous (midpoint)** — derived from "Average daily sitting time (including work, commuting, and leisure)".

Ordinal categories mapped to midpoint hours:

| Response | Midpoint (hrs) |
|---|---|
| Less than 4 hours | 2 |
| 4–6 hours | 5 |
| 6–8 hours | 7 |
| More than 8 hours | 9 |

In [101]:
SITTING_TIME_MAP = {
    'Less than 4 hours': 2,
    '4\u20136 hours':    5,
    '6\u20138 hours':    7,
    'More than 8 hours': 9,
}

df['sitting_time'] = df['daily_sitting_raw'].map(SITTING_TIME_MAP)

print(df['sitting_time'].value_counts().sort_index())
print()


sitting_time
5     5
7    13
9     7
Name: count, dtype: int64



## Early-life mobility-focused activity
**Nominal** — "Which type of training has influenced your body the most over your lifetime (especially while growing up)?"

5 categories kept as-is with shortened labels:

| Raw response | Stored as |
|---|---|
| Flexibility-based (dance, yoga, gymnastics) | Flexibility-based |
| Endurance-based (running, cycling) | Endurance-based |
| Strength-based (weights, power sports) | Strength-based |
| Mixed / hybrid | Mixed / hybrid |
| No clear dominant influence | No clear dominant |

In [102]:
EARLY_LIFE_MAP = {
    'Flexibility-based (dance, yoga, gymnastics)': 'Flexibility-based',
    'Endurance-based (running, cycling)':          'Endurance-based',
    'Strength-based (weights, power sports)':      'Strength-based',
    'Mixed / hybrid':                              'Mixed / hybrid',
    'No clear dominant influence':                 'No clear dominant',
}

df['early_life_training'] = df['dominant_training_raw'].map(EARLY_LIFE_MAP)

print(df['early_life_training'].value_counts())
print()
print('Unmapped (NaN):', df['early_life_training'].isna().sum())
print()


early_life_training
Flexibility-based    9
Mixed / hybrid       8
Endurance-based      5
Strength-based       2
No clear dominant    1
Name: count, dtype: int64

Unmapped (NaN): 0



## Save clean survey data

In [103]:
# Rename raw columns to clean analysis names before saving
df = df.rename(columns={
    'age_raw':               'age_group',
    'gender_raw':            'gender',
    'end_range_control_raw': 'end_range_control',
    'hours_per_week_raw':    'weekly_volume',
    'injury_raw':            'injury_presence',
    'chronic_pain_raw':      'chronic_pain',
    'sleep_duration_raw':    'sleep_duration',
})

COLS_TO_SAVE = [
    'name',
    'age_group',             # raw text label, e.g. "18-29"
    'gender',                # raw text label
    'bmi',                   # computed continuous
    'training_category',     # derived: Low Active / Hybrid / Strength-dominant / Flexibility-dominant / Unclassified
    'early_life_training',   # nominal: early-life dominant training background
    'end_range_control',     # raw text label
    'weekly_volume',         # raw text label
    'injury_presence',       # raw text label (Yes/No)
    'chronic_pain',          # raw text label
    'sleep_duration',        # raw text label
    'sleep_quality',         # computed composite 0-10
    'physical_recovery',     # frequency of physical recovery practices 1-6
    'mental_recovery',       # frequency of mental recovery practices 1-6
    'stress_score',          # computed composite 0-4
    'sitting_time',          # average daily sitting hours (midpoint): 5, 7, or 9
    'sedentary_score',       # computed composite -5 to 11
]

df[COLS_TO_SAVE].to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df[COLS_TO_SAVE].shape}')
print()
print('Missing values:')
print(df[COLS_TO_SAVE].isnull().sum())

Saved: ../data/processed/combined_data.csv
Shape: (25, 17)

Missing values:
name                   0
age_group              0
gender                 0
bmi                    0
training_category      0
early_life_training    0
end_range_control      0
weekly_volume          0
injury_presence        0
chronic_pain           0
sleep_duration         0
sleep_quality          0
physical_recovery      0
mental_recovery        0
stress_score           0
sitting_time           0
sedentary_score        0
dtype: int64
